In [1]:
from pathlib import Path
import polars as pl

# Location where training exports parquet files
MODELS_DIR = Path("models")


def latest_parquet(pattern: str) -> Path | None:
    files = sorted(MODELS_DIR.glob(pattern), key=lambda p: p.stat().st_mtime)
    return files[-1] if files else None


def load_optional(path: Path | None, label: str) -> pl.DataFrame | None:
    if path is None:
        print(f"No {label} parquet found in {MODELS_DIR.resolve()}")
        return None

    print(f"Loading {label}: {path}")
    df = pl.read_parquet(path)
    print(f"{label} rows: {df.height:,} | cols: {df.width}")
    return df


latest_events = latest_parquet("*_events.parquet")
latest_anomalies = latest_parquet("*_anomalies.parquet")

events_df = load_optional(latest_events, "events")
anomalies_df = load_optional(latest_anomalies, "anomalies")

if events_df is not None:
    print("\nEvents preview:")
    print(events_df.head(10))

if anomalies_df is not None:
    print("\nAnomalies preview:")
    print(anomalies_df.select(["time", "symbol", "trade_condition", "trade_size", "anomaly_score"]).head(10))

# Quick sanity summaries
if events_df is not None and "root" in events_df.columns:
    print("\nEvents by ticker:")
    print(events_df.group_by("root").len().sort("len", descending=True))

if anomalies_df is not None and "root" in anomalies_df.columns:
    print("\nAnomalies by ticker:")
    print(anomalies_df.group_by("root").len().sort("len", descending=True))

Loading events: models/NVDA_2026-04-13_events.parquet
events rows: 792 | cols: 14
Loading anomalies: models/NVDA_2026-04-13_anomalies.parquet
anomalies rows: 28,803 | cols: 33

Events preview:
shape: (10, 14)
┌──────────┬──────┬────────────┬────────────┬───┬────────────┬────────────┬────────────┬───────────┐
│ event_id ┆ root ┆ symbol     ┆ expiration ┆ … ┆ total_size ┆ mean_anoma ┆ worst_anom ┆ dominant_ │
│ ---      ┆ ---  ┆ ---        ┆ ---        ┆   ┆ ---        ┆ ly_score   ┆ aly_score  ┆ trade_con │
│ u32      ┆ str  ┆ str        ┆ datetime[μ ┆   ┆ f64        ┆ ---        ┆ ---        ┆ dition    │
│          ┆      ┆            ┆ s]         ┆   ┆            ┆ f64        ┆ f64        ┆ ---       │
│          ┆      ┆            ┆            ┆   ┆            ┆            ┆            ┆ str       │
╞══════════╪══════╪════════════╪════════════╪═══╪════════════╪════════════╪════════════╪═══════════╡
│ 1        ┆ NVDA ┆ NVDA260209 ┆ 2026-02-09 ┆ … ┆ 116.0      ┆ -0.012032  ┆ -0.01948